<a href="https://colab.research.google.com/github/SFStefenon/IAAMASP/blob/main/IAAMASP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.6/275.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.8 MB/s eta 0:00:00


In [7]:
horizon = 10
epochs = 1
data_split = 0.2 # Test (%)
max_steps = epochs
input_size = horizon
freq = "s"

import numpy as np
import pandas as pd
import time
from statsmodels.tools.eval_measures import rmse
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_log_error
from neuralforecast import NeuralForecast
from neuralforecast.models import MLP, TFT, RNN, DilatedRNN, NHITS, TCN, BiTCN, LSTM, NBEATS, NBEATSx, GRU, Informer, TiDE, PatchTST, FEDformer, DeepAR, TimesNet

# Preprocessing
df = pd.read_csv('miris_load.csv', parse_dates=['DateTime'])
df = df.rename(columns={'Conso': 'Load'})
time_df = df['DateTime']
x = np.asarray(df["Load"], dtype=float)
time_data = pd.date_range(start=str(time_df[0]), periods=len(x), freq=freq)

# Create the DataFrame
df = pd.DataFrame({"unique_id": ["Airline1"] * len(x), "ds": time_data, "y": x,"trend": np.arange(len(x), dtype=float)})

# Split data based on your criteria
split_point = int(len(df) * (1 - data_split))
Y_train_df = df.iloc[:split_point].reset_index(drop=True)
Y_test_df = df.iloc[split_point:].reset_index(drop=True)
if len(Y_test_df) < horizon:
    raise ValueError("The test set is smaller than the forecast horizon.")
if len(Y_train_df) < input_size:
    raise ValueError("The training set is smaller than the input size.")
y_true = Y_test_df["y"].iloc[:horizon].to_numpy(dtype=float)

def performance(y_true, y_pred, time_s):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse_v = rmse(y_true, y_pred)
    mae_v = mean_absolute_error(y_true, y_pred)
    mape_v = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-10)))
    msle_v = mean_squared_log_error(np.clip(y_true, 0, None), np.clip(y_pred, 0, None))
    return f"{rmse_v:.2E} & {mae_v:.2E} & {mape_v:.2E} & {msle_v:.2E} & {time_s:.2E} \\\\"

def evaluate_model(model_class):
    start = time.time()
    nf = NeuralForecast(models=[model_class(input_size=input_size, h=horizon, max_steps=max_steps)],freq=freq)
    nf.fit(df=Y_train_df)
    Y_hat_df = nf.predict().reset_index()
    time_s = time.time() - start
    model_name = model_class.__name__
    y_pred = Y_hat_df[model_name].iloc[:horizon].to_numpy(dtype=float)
    return f"{model_name} & {performance(y_true, y_pred, time_s)}"

models_list = [MLP, TFT, RNN, DilatedRNN, NHITS, TCN, BiTCN, LSTM, NBEATS, NBEATSx, GRU, Informer, TiDE, PatchTST, FEDformer, DeepAR, TimesNet]
results = [evaluate_model(model_class) for model_class in models_list]

INFO:lightning_fabric.utilities.seed:Seed set to 1
/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:591: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(val_check_interval=1)` was configured so validation will run after every batch.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ mlp          │ ModuleList    │  1.1 M │ train │     0 │
│ 4 │ out          │ Linear        │ 10.2 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4                                                                          
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(val_check_interval=1)` was configured so validation will run after every batch.


Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


MLP & 1.33E-01 & 1.26E-01 & 1.34E+00 & 1.29E-02 & 1.77E+01 \\
TFT & 8.49E-03 & 8.35E-03 & 8.89E-02 & 6.07E-05 & 8.85E+00 \\
RNN & 8.64E-03 & 8.51E-03 & 9.05E-02 & 6.28E-05 & 9.17E+00 \\
DilatedRNN & 8.64E-03 & 8.50E-03 & 9.05E-02 & 6.29E-05 & 6.70E+00 \\
NHITS & 2.22E-01 & 1.91E-01 & 2.03E+00 & 1.46E-02 & 5.94E+00 \\
TCN & 8.70E-03 & 8.57E-03 & 9.12E-02 & 6.38E-05 & 7.02E+00 \\
BiTCN & 3.41E-01 & 3.40E-01 & 3.63E+00 & 8.03E-03 & 5.34E+00 \\
LSTM & 8.66E-03 & 8.52E-03 & 9.07E-02 & 6.31E-05 & 6.84E+00 \\
NBEATS & 1.61E-01 & 1.29E-01 & 1.38E+00 & 1.81E-02 & 5.52E+00 \\
NBEATSx & 1.61E-01 & 1.29E-01 & 1.38E+00 & 1.81E-02 & 6.14E+00 \\
GRU & 8.66E-03 & 8.53E-03 & 9.08E-02 & 6.32E-05 & 6.21E+00 \\
Informer & 5.87E-01 & 4.65E-01 & 4.97E+00 & 7.21E-03 & 7.38E+00 \\
TiDE & 2.86E-01 & 2.58E-01 & 2.75E+00 & 7.31E-03 & 5.54E+00 \\
PatchTST & 3.36E-03 & 2.95E-03 & 3.15E-02 & 9.48E-06 & 6.14E+00 \\
FEDformer & 6.76E-02 & 5.73E-02 & 6.07E-01 & 3.52E-03 & 8.45E+00 \\
DeepAR & 1.18E-01 & 1.18E-01 & 1.2

In [10]:
for result in results:
    print(result)

MLP & 1.33E-01 & 1.26E-01 & 1.34E+00 & 1.29E-02 & 1.77E+01 \\
TFT & 8.49E-03 & 8.35E-03 & 8.89E-02 & 6.07E-05 & 8.85E+00 \\
RNN & 8.64E-03 & 8.51E-03 & 9.05E-02 & 6.28E-05 & 9.17E+00 \\
DilatedRNN & 8.64E-03 & 8.50E-03 & 9.05E-02 & 6.29E-05 & 6.70E+00 \\
NHITS & 2.22E-01 & 1.91E-01 & 2.03E+00 & 1.46E-02 & 5.94E+00 \\
TCN & 8.70E-03 & 8.57E-03 & 9.12E-02 & 6.38E-05 & 7.02E+00 \\
BiTCN & 3.41E-01 & 3.40E-01 & 3.63E+00 & 8.03E-03 & 5.34E+00 \\
LSTM & 8.66E-03 & 8.52E-03 & 9.07E-02 & 6.31E-05 & 6.84E+00 \\
NBEATS & 1.61E-01 & 1.29E-01 & 1.38E+00 & 1.81E-02 & 5.52E+00 \\
NBEATSx & 1.61E-01 & 1.29E-01 & 1.38E+00 & 1.81E-02 & 6.14E+00 \\
GRU & 8.66E-03 & 8.53E-03 & 9.08E-02 & 6.32E-05 & 6.21E+00 \\
Informer & 5.87E-01 & 4.65E-01 & 4.97E+00 & 7.21E-03 & 7.38E+00 \\
TiDE & 2.86E-01 & 2.58E-01 & 2.75E+00 & 7.31E-03 & 5.54E+00 \\
PatchTST & 3.36E-03 & 2.95E-03 & 3.15E-02 & 9.48E-06 & 6.14E+00 \\
FEDformer & 6.76E-02 & 5.73E-02 & 6.07E-01 & 3.52E-03 & 8.45E+00 \\
DeepAR & 1.18E-01 & 1.18E-01 & 1.2